# 工单 04／05 数据可用性诊断的计数与来源核查

使用已存在的 Thesis 解释器，从项目根目录运行。仅使用标准库，无需安装 Notebook 内核；本轮已用 `conda run -p ...\.conda\envs\Thesis python` 逐个执行以下代码单元。代码不调用正式筛选，不重新提取评论，不确认工单 06。

逐条语义记录是拟议判断；41 条非概率代表案例不能外推全部待裁决集合。以下只验证计数、来源定位、字节摘要和旧文件未变，不能用程序断言替代语义复核。


In [ ]:
from collections import Counter
from datetime import datetime, timezone
import gzip
import hashlib
import json
from pathlib import Path
import sys

ROOT = Path(r'C:\Users\Kane\Documents\CodexProject\Thesis')
assert Path(sys.executable).resolve() == (ROOT / '.conda/envs/Thesis/python.exe').resolve()

def load(relative):
    return json.loads((ROOT / relative).read_text(encoding='utf-8'))

def sha(path):
    with path.open('rb') as stream:
        return hashlib.file_digest(stream, 'sha256').hexdigest()

summary = load('.scratch/data-preparation/verification/usability-04-05-draft-01.json')
catalog = load('records/catalog/grain-A-1.0.6.json')
source = load('records/sources/amazon-2023-grocery-metadata.json')
cases = load(summary['representative_path'])
notes = load('.scratch/data-preparation/diagnostics/usability-case-review-20261012.json')
assert sha(ROOT / summary['representative_path']) == summary['representative_sha256']
assert sha(ROOT / summary['profile_path']) == summary['profile_sha256']
assert len(cases) == len(notes['annotations']) == 41
case_map = {row['parent_asin']: row for row in cases}
assert len(case_map) == 41
assert set(case_map) == {row['parent_asin'] for row in notes['annotations']}
groups = Counter()
for row in notes['annotations']:
    sample = case_map[row['parent_asin']]
    for key in ('source_line', 'source_line_sha256', 'source_sha256', 'existing_reason'):
        assert row[key] == sample[key]
    assert row['source_title'] == sample['metadata']['title']
    assert row['status'] == 'proposed_only_not_formal_adjudication'
    groups[row['group']] += 1
assert groups == {'I': 5, 'P': 4, 'X': 24, 'C': 5, 'U': 3}
print(dict(groups))


In [ ]:
pending_ids = {}
pending_counts = Counter()
pending_ingredient_lexemes = 0
import re
ingredient_terms = re.compile(r'\bingredients?\b|\bcontains only\b|\bnothing else is added\b|\bmade (?:from|of) only\b|配料|原料', re.I)
with (ROOT / catalog['pending_path']).open(encoding='utf-8') as stream:
    for line in stream:
        row = json.loads(line)
        parent = row['parent_asin']
        assert parent not in pending_ids
        pending_ids[parent] = (row['source_line'], row['reason'])
        pending_counts[row['reason']] += 1
        if row['reason'] == 'missing_composition':
            item = row['metadata']
            texts = [item.get('title', ''), *item.get('features', []), *item.get('description', []), *item.get('details', {}).values()]
            pending_ingredient_lexemes += any(isinstance(text, str) and ingredient_terms.search(text) for text in texts)
profile_counts = Counter()
profile_ids = set()
priority_count = 0
with (ROOT / summary['profile_path']).open(encoding='utf-8') as stream:
    for line in stream:
        row = json.loads(line)
        parent = row['parent_asin']
        assert parent not in profile_ids
        profile_ids.add(parent)
        assert pending_ids[parent] == (row['source_line'], row['existing_reason'])
        assert row['new_scope_decision'] == 'not_made'
        assert row['review_admission'] == 'not_evaluated'
        profile_counts[row['existing_reason']] += 1
        flags = row['flags']
        if row['existing_reason'] == 'missing_composition':
            priority_count += flags['single_title_crop_family'] and flags['explicit_physical_word_in_title'] and not flags['non_scope_object_word_in_title']
assert profile_ids == set(pending_ids)
assert dict(pending_counts) == dict(profile_counts) == summary['pending_reasons']
assert len(profile_ids) == 56209 and sum(pending_counts.values()) == 56209
assert pending_ingredient_lexemes == 555
assert priority_count == summary['composition_gate']['single_crop_physical_no_non_scope_title_signal'] == 1367
print({'pending': len(profile_ids), 'missing_composition': pending_counts['missing_composition'], 'ingredient_lexemes': pending_ingredient_lexemes, 'priority_leads_not_admissions': priority_count})


In [ ]:
by_line = {row['source_line']: row for row in cases}
matched = set()
with gzip.open(ROOT / source['raw_path'], 'rb') as stream:
    for line_number, raw_line in enumerate(stream, 1):
        if line_number in by_line:
            sample = by_line[line_number]
            assert hashlib.sha256(raw_line).hexdigest() == sample['source_line_sha256']
            assert json.loads(raw_line) == sample['metadata']
            matched.add(sample['parent_asin'])
assert line_number == summary['source_rows'] == 603274
assert matched == set(case_map)
assert sha(ROOT / source['raw_path']) == summary['source_sha256']
assert all(sha(ROOT / path) == value for path, value in summary['preserved_sha256'].items())
for name in ('catalog', 'audit', 'pending'):
    assert sha(ROOT / catalog[name + '_path']) == catalog[name + '_sha256']
reviews = load('records/reviews/grain-A-1.0.6-review-diagnostics-1.0.0.json')
assert sha(ROOT / reviews['extract_path']) == reviews['extract_sha256']
review_count = 0
accounts = set()
with (ROOT / reviews['extract_path']).open(encoding='utf-8') as stream:
    for line in stream:
        review = json.loads(line)
        review_count += 1
        accounts.add(review['user_id'])
assert review_count == reviews['raw']['reviews'] == 260
assert len(accounts) == reviews['raw']['unique_accounts'] == 256
originals = load('.scratch/data-preparation/verification/original-materials.json')
assert len(originals['files']) == 26
assert all(sha(ROOT / row['path']) == row['sha256'] for row in originals['files'])
print({'metadata_eof': line_number, 'samples_bound_to_original_lines': len(matched), 'protected_files': len(summary['preserved_sha256']), 'originals_unchanged': 26, 'existing_reviews_unchanged': review_count, 'existing_accounts': len(accounts)})


In [ ]:
result = {
    'status': 'independent_diagnostic_count_and_provenance_checks_passed',
    'executed_at_utc': datetime.now(timezone.utc).isoformat(),
    'interpreter': sys.executable,
    'python': sys.version,
    'metadata_eof': line_number,
    'profile_rows': len(profile_ids),
    'pending_reasons': dict(pending_counts),
    'semantic_case_primary_groups': dict(groups),
    'semantic_cases_reviewed': len(case_map),
    'semantic_population_not_individually_reviewed': len(profile_ids) - len(case_map),
    'priority_leads_not_admissions': priority_count,
    'existing_reviews': review_count,
    'existing_accounts': len(accounts),
    'preserved_sha256': summary['preserved_sha256'],
    'original_materials_unchanged': True,
    'original_material_count': len(originals['files']),
    'inputs': {path: sha(ROOT / path) for path in (
        '.scratch/data-preparation/verification/usability-04-05-draft-01.json',
        '.scratch/data-preparation/diagnostics/usability-case-review-20261012.json',
        '.scratch/data-preparation/diagnostics/usability-diagnosis-20261012.ipynb',
    )},
    'limitations': [
        'Independent counts and byte bindings do not replace semantic judgment.',
        'The 41 non-probability cases cannot estimate full-population semantic counts.',
        'No formal rule, scope decision, review extraction or user qualification was changed.',
    ],
}
destination = ROOT / '.scratch/data-preparation/verification/usability-04-05-draft-01-check.json'
destination.write_text(json.dumps(result, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
print(result['status'])
